In [4]:
import datasets
from IPython.display import display, HTML

messirve_test = datasets.load_dataset("spanish-ir/messirve", "full", revision="1.2", split="test")
corpus = datasets.load_dataset("spanish-ir/eswiki_20240401_corpus", split="corpus")

print(f"Test: {len(messirve_test):,} rows | Corpus: {len(corpus):,} documents")

Test: 174,078 rows | Corpus: 14,047,759 documents


In [7]:
def show_query(query_id: int, max_doc_chars: int = 1500):
    """Display a query and all its relevant documents."""
    rows = messirve_test.filter(
        lambda b: [x == query_id for x in b["id"]], batched=True, num_proc=32
    )
    if len(rows) == 0:
        print(f"Query ID {query_id} not found in test set.")
        return

    query_text = rows[0]["query"]
    rel_doc_ids = set(rows["docid"])

    docs = corpus.filter(
        lambda b: [x in rel_doc_ids for x in b["docid"]], batched=True, num_proc=32
    )

    html = f"""
    <div style="font-family: sans-serif; max-width: 800px;">
    <h3 style="color: #2c3e50;">Query <code>{query_id}</code></h3>
    <div style="background: #eaf2f8; padding: 12px 16px; border-left: 4px solid #2980b9;
                border-radius: 4px; font-size: 16px; margin-bottom: 20px;">
        {query_text}
    </div>
    <p style="color: #7f8c8d;">{len(rel_doc_ids)} relevant document(s)</p>
    """

    for i in range(len(docs)):
        doc = docs[i]
        title = doc.get("title", "") or ""
        text = doc.get("text", "") or ""
        snippet = text[:max_doc_chars] + ("..." if len(text) > max_doc_chars else "")

        html += f"""
        <div style="border: 1px solid #ddd; border-radius: 6px; padding: 12px 16px;
                    margin-bottom: 12px; background: #fdfefe;">
        <div style="font-size: 11px; color: #95a5a6;">Document {i+1} &mdash; ID {doc["docid"]}</div>
        <div style="font-weight: bold; font-size: 14px; color: #2c3e50; margin: 4px 0;">
            {title}
        </div>
        <div style="font-size: 13px; color: #34495e; line-height: 1.5;">
            {snippet}
        </div>
        </div>
        """

    html += "</div>"
    display(HTML(html))

In [15]:
import random

# show_query(12345)                                    # specific query ID
show_query(random.choice(messirve_test["id"]))       # random query

Filter (num_proc=32):   0%|          | 0/174078 [00:00<?, ? examples/s]

Filter (num_proc=32): 100%|██████████| 14047759/14047759 [00:01<00:00, 7508913.15 examples/s] 
